In [ ]:
import os, json, time
from google import genai
from dotenv import load_dotenv
import mubit, mubit.learn

load_dotenv()

SESSION = f"learn-demo-{int(time.time())}"
GEMINI_MODEL = "gemini-3.1-flash-lite-preview"
pp = lambda obj: print(json.dumps(obj, indent=2, default=str))

# mubit.learn.init() wraps all supported LLM clients (OpenAI, Anthropic,
# Google GenAI, LiteLLM). After this, every LLM call automatically gets
# memory capabilities — lesson injection before the call, trace capture
# and knowledge extraction after the call.
run = mubit.learn.init(
    api_key=os.environ["MUBIT_API_KEY"],
    endpoint=os.environ.get("MUBIT_ENDPOINT", "http://127.0.0.1:3000"),
    agent_id="demo-agent",
    session_id=SESSION,
    inject_lessons=True,
    auto_reflect=True,
    auto_extract=True,
    max_token_budget=800,
    cache_ttl_seconds=5,
)

# This client is automatically wrapped by mubit.learn.
# Under the hood, generate_content() is now:
#   1. pre-call:  fetch lessons from Mubit → inject into prompt
#   2. call:      normal Gemini API call
#   3. post-call: capture interaction + extract rules/lessons → ingest to Mubit
llm = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print(f"Session: {SESSION}")
print(f"LLM: {GEMINI_MODEL}")
print("Ready — all Gemini calls are now auto-instrumented.")

In [ ]:
# Call #1: No lessons exist yet.
# Pre-call:  mubit.learn calls get_context() but finds nothing to inject.
# Post-call: mubit.learn captures the interaction and scans the response for
#            rules ("always", "never"), lessons ("the fix was", "caused by"),
#            and facts. All extracted items are sent to Mubit in the background.
response_1 = llm.models.generate_content(
    model=GEMINI_MODEL,
    contents=(
        "You are a senior SRE investigating a production incident. "
        "A developer rotated the JWT signing key but forgot to flush the Redis token cache first. "
        "As a result, stale tokens signed with the old key were served from Redis for 5 minutes, "
        "causing widespread auth failures.\n\n"
        "Write a concise post-mortem summary. Include:\n"
        "- Root cause\n"
        "- What the fix was\n"
        "- Rules to prevent this in future (use 'always' and 'never' phrasing)\n"
        "Keep it under 200 words."
    ),
)
print(response_1.text)

In [ ]:
# Wait for the background IngestWorker to send items to Mubit
# and for Mubit to embed + index them.
time.sleep(8)

query_client = mubit.Client(
    endpoint=os.environ.get("MUBIT_ENDPOINT", "http://127.0.0.1:3000"),
    api_key=os.environ["MUBIT_API_KEY"],
    run_id=SESSION,
)
query_client.set_transport("http")

health = query_client.memory_health(session_id=SESSION, limit=100)
print(f"Memory after call #1: {health.get('entry_counts', {})}")
print("\nAll auto-extracted — zero manual calls.")

## Call #2 — Different question, lessons auto-injected

Pre-call: mubit.learn calls get_context() and finds the rules/lessons extracted from call #1. It prepends them to the prompt inside `<memory_context>` tags. The LLM now has context from the previous call without any manual retrieval code.

Post-call: same auto-capture and extraction as call #1.

In [ ]:
# Clear the lesson cache so mubit.learn fetches fresh context from Mubit
# (rather than using the cached empty result from call #1)
mubit.learn._lesson_cache.clear()

response_2 = llm.models.generate_content(
    model=GEMINI_MODEL,
    contents="I need to rotate our JWT signing keys this weekend. What steps should I follow to avoid any downtime?",
)
print(response_2.text)

In [ ]:
# run.end() triggers Mubit's reflection agent, which analyzes all traces
# in the session and distills higher-order lessons. These lessons persist
# and will be injected into future sessions automatically.
run.end()
time.sleep(5)

health = query_client.memory_health(session_id=SESSION, limit=100)
print(f"Final memory: {health.get('entry_counts', {})}")

lessons = query_client.control.lessons({"run_id": SESSION, "limit": 10})
for l in lessons.get("lessons", []):
    print(f"  [{l.get('lesson_type', '?'):12s}] {l.get('content', '')[:120]}...")

---

**Total Mubit-specific code: `mubit.learn.init(...)` — one line.**

Everything else — extraction, injection, reflection — was automatic.